In [1]:
#Cell 1 — Imports & setup:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, 
                             precision_recall_curve, average_precision_score)
import xgboost as xgb
import shap
import mlflow
import mlflow.xgboost

os.chdir('/Users/niravparmar/retail-fraud-detection')
pd.set_option('display.float_format', lambda x: '%.4f' % x)
plt.style.use('seaborn-v0_8')

print("All imports successful.")
print(f"XGBoost version: {xgb.__version__}")
print(f"SHAP version:    {shap.__version__}")

All imports successful.
XGBoost version: 3.2.0
SHAP version:    0.51.0


In [2]:
#Cell 2 — Load feature matrix:

print("Loading features...")
X = pd.read_csv('data/processed/features.csv')
y = pd.read_csv('data/processed/target.csv').squeeze()

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Fraud rate:     {y.mean()*100:.2f}%")
print(f"\nFeature columns:")
print(list(X.columns))

Loading features...
Features shape: (590540, 38)
Target shape:   (590540,)
Fraud rate:     3.50%

Feature columns:
['hour', 'day_of_week', 'day_of_month', 'week', 'is_night', 'is_weekend', 'TransactionAmt', 'amt_log', 'is_round_amount', 'amt_deviation', 'card1_amt_mean', 'card1_amt_std', 'card1_count', 'email_count', 'email_fraud_rate', 'ProductCD_enc', 'card4_enc', 'card6_enc', 'email_domain_enc', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5']


In [ ]:
#Cell 3 — Train/test split:

# Split data — 80% train, 20% test
# Stratify ensures same fraud rate in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # critical for imbalanced data
)

print("Data split complete:")
print(f"  Training set:   {X_train.shape[0]:,} transactions")
print(f"  Test set:       {X_test.shape[0]:,} transactions")
print(f"  Train fraud:    {y_train.mean()*100:.2f}%")
print(f"  Test fraud:     {y_test.mean()*100:.2f}%")

# Calculate class weight for imbalanced data
fraud_count = y_train.sum()
legit_count = len(y_train) - fraud_count
scale_pos_weight = legit_count / fraud_count

print(f"\nClass imbalance ratio: {scale_pos_weight:.1f}x")
print(f"  Legitimate: {legit_count:,}")
print(f"  Fraud:      {fraud_count:,}")

In [ ]:
#Cell 4 — Train XGBoost model with MLflow tracking:

# Start MLflow experiment
mlflow.set_experiment("fraud_detection_xgboost")

print("Training XGBoost model...")
print("MLflow tracking started...")

with mlflow.start_run(run_name="xgboost_baseline"):
    
    # Model parameters
    params = {
        'n_estimators':     500,
        'max_depth':        6,
        'learning_rate':    0.05,
        'subsample':        0.8,
        'colsample_bytree': 0.8,
        'scale_pos_weight': scale_pos_weight,
        'eval_metric':      'auc',
        'random_state':     42,
        'n_jobs':           -1
    }
    
    # Log parameters to MLflow
    mlflow.log_params(params)
    
    # Train model
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100
    )
    
    # Predictions
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred       = (y_pred_proba >= 0.5).astype(int)
    
    # Metrics
    auc_roc = roc_auc_score(y_test, y_pred_proba)
    avg_precision = average_precision_score(y_test, y_pred_proba)
    
    # Log metrics to MLflow
    mlflow.log_metric("auc_roc", auc_roc)
    mlflow.log_metric("avg_precision", avg_precision)
    
    # Log model
    mlflow.xgboost.log_model(model, "xgboost_model")
    
    print(f"\n{'='*40}")
    print(f"MODEL TRAINING COMPLETE")
    print(f"{'='*40}")
    print(f"AUC-ROC Score:     {auc_roc:.4f}")
    print(f"Avg Precision:     {avg_precision:.4f}")
    print(f"{'='*40}")

In [ ]:
#Cell 5 — Full model evaluation:

# Detailed performance report
print("=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, 
      target_names=['Legitimate', 'Fraud']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\n=== CONFUSION MATRIX ===")
print(f"True Negatives  (Correctly caught legitimate): {tn:,}")
print(f"False Positives (Legitimate flagged as fraud): {fp:,}")
print(f"True Positives  (Correctly caught fraud):      {tp:,}")
print(f"False Negatives (Fraud missed by model):       {fn:,}")
print(f"\nFraud caught:  {tp/(tp+fn)*100:.1f}%")
print(f"False alarms:  {fp/(fp+tn)*100:.1f}%")

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Fraud'],
            yticklabels=['Legitimate', 'Fraud'])
plt.title('Confusion Matrix — XGBoost Fraud Detector', 
          fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('reports/figures/08_confusion_matrix.png', 
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Cell 6 — ROC Curve:

# ROC curve shows model performance across all thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#F44336', linewidth=2,
             label=f'XGBoost (AUC = {auc_roc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#F44336')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall, precision, color='#2196F3', linewidth=2,
             label=f'XGBoost (AP = {avg_precision:.4f})')
axes[1].axhline(y=y_test.mean(), color='k', linestyle='--',
                label=f'Baseline ({y_test.mean():.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/figures/09_roc_pr_curves.png', 
            dpi=150, bbox_inches='tight')
plt.show()

print(f"AUC-ROC: {auc_roc:.4f} — {'Excellent' if auc_roc > 0.9 else 'Good' if auc_roc > 0.8 else 'Needs improvement'}")

In [ ]:
#Cell 7 — Feature importance:

# Which features matter most to the model?
importance_df = pd.DataFrame({
    'feature':    X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== TOP 20 MOST IMPORTANT FEATURES ===")
print(importance_df.head(20).to_string(index=False))

# Plot
plt.figure(figsize=(12, 8))
top20 = importance_df.head(20)
colors = ['#F44336' if i < 5 else '#FF9800' if i < 10 else '#2196F3' 
          for i in range(len(top20))]
plt.barh(top20['feature'], top20['importance'], color=colors)
plt.xlabel('Feature Importance Score')
plt.title('Top 20 Most Important Features — XGBoost', 
          fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('reports/figures/10_feature_importance.png', 
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Cell 8 — SHAP values (the magic):

# SHAP explains WHY the model makes each prediction
# This is what makes your project production-ready

print("Calculating SHAP values...")
print("This takes 2-3 minutes...")

# Use a sample for speed
X_sample = X_test.sample(1000, random_state=42)

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# SHAP Summary plot — most important chart in your project
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, 
                  plot_type="bar",
                  show=False)
plt.title('SHAP Feature Importance — Global', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/11_shap_importance.png', 
            dpi=150, bbox_inches='tight')
plt.show()

# SHAP Beeswarm — shows direction of impact
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reports/figures/12_shap_beeswarm.png', 
            dpi=150, bbox_inches='tight')
plt.show()

print("\nSHAP analysis complete.")

In [ ]:
#Cell 9 — Explain a single fraud prediction:

# Pick one real fraud transaction and explain exactly why
# the model flagged it — this is your demo feature

fraud_indices = X_test[y_test == 1].index
first_fraud_idx = fraud_indices[0]
first_fraud = X_test.loc[[first_fraud_idx]]

fraud_proba = model.predict_proba(first_fraud)[0][1]

print(f"=== SINGLE FRAUD EXPLANATION ===")
print(f"Transaction index: {first_fraud_idx}")
print(f"Fraud probability: {fraud_proba*100:.1f}%")
print(f"Model decision:    {'FRAUD' if fraud_proba > 0.5 else 'LEGITIMATE'}")
print(f"\nTop features for this transaction:")
print(first_fraud.T.rename(columns={first_fraud_idx: 'value'}).head(10))

# SHAP waterfall for this single transaction
shap_single = explainer.shap_values(first_fraud)
shap.waterfall_plot(
    shap.Explanation(
        values=shap_single[0],
        base_values=explainer.expected_value,
        data=first_fraud.values[0],
        feature_names=list(X_test.columns)
    )
)

In [ ]:
#Cell 10 — Save model & final summary:

import pickle

# Save model
os.makedirs('src/models', exist_ok=True)
with open('src/models/xgboost_fraud_detector.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved to src/models/xgboost_fraud_detector.pkl")
print("")
print("="*50)
print("DAY 4 COMPLETE — FRAUD DETECTOR BUILT")
print("="*50)
print(f"\nModel Performance:")
print(f"  AUC-ROC Score:  {auc_roc:.4f}")
print(f"  Avg Precision:  {avg_precision:.4f}")
print(f"  Fraud caught:   {tp/(tp+fn)*100:.1f}%")
print(f"  False alarms:   {fp/(fp+tn)*100:.1f}%")
print(f"\nArtifacts saved:")
print(f"  Model:   src/models/xgboost_fraud_detector.pkl")
print(f"  Charts:  reports/figures/08 to 12")
print(f"  MLflow:  mlflow_runs/")
print(f"\nDay 5 → Anomaly detection with Isolation Forest")